# Flatland
Docs at: https://flatland.aicrowd.com/intro.html


## File structure


```
├── Notebooks, Readme, packages ..
├── agents: RL agents implementation
│   ├── curiosity.py
│   ├── dqn.py
│   ├── per.py
│   ├── qlearning.py
│   └── random.py
├── utils: Helpers to train, test, inspect agents
│   ├── fast_tree_obs.py
│   ├── logging.py
│   ├── rl_helpers.py
└── └── utils.py
```



## Imports
First we need to import all libraries and clone external helper code

In [ ]:
#@title << Setup Google Colab by running this cell {display-mode: "form"}
import sys
if 'google.colab' in sys.modules:
    !rm -rf rl-on-trains-workshop
    # Clone GitHub repository
    !git clone --single-branch --branch main https://github.com/YanickSchraner/rl-on-trains-workshop

    # Copy files required to run the code
    !mkdir rl-on-trains-workshop/agents
    !cp -r "rl-on-trains-workshop/utils" "rl-on-trains-workshop/agents" .

    # Install packages via pip
    !pip install flatland-rl==3.0.15


In [ ]:
# Import libraries
import numpy as np

# In Flatland you can use custom observation builders and predicitors
# Observation builders generate the observation needed by the controller
# Preditctors can be used to do short time prediction which can help in avoiding conflicts in the network

# First of all we import the Flatland rail environment
from flatland.envs.rail_env import RailEnv, RailEnvActions
from flatland.envs.rail_generators import sparse_rail_generator
from flatland.envs.malfunction_generators import MalfunctionParameters, ParamMalfunctionGen
from utils.observation_utils import normalize_observation
from flatland.envs.observations import TreeObsForRailEnv
from flatland.envs.line_generators import sparse_line_generator

# We also include a renderer because we want to visualize what is going on in the environment
from flatland.utils.rendertools import RenderTool

In [ ]:
%%capture
%matplotlib inline

# Helper function to visualize an episode

import matplotlib.pyplot as plt
import matplotlib.animation
plt.rcParams["animation.html"] = "jshtml"

def display_episode(frames):
    fig, ax = plt.subplots(figsize=(12,12))
    imgplot = plt.imshow(frames[0])
    def animate(i):
        imgplot.set_data(frames[i])
    animation = matplotlib.animation.FuncAnimation(fig, animate, frames=len(frames))
    return animation

## Presentation of the RL environment

In the next cells we specify the characteristics of our flatland grid and the trains.
We use the `TreeObs` observation for all of our agents.

Training on simple small tasks is the best way to get familiar with the environment.
We started by importing the necessary rail and schedule generators.
The rail generator will generate the railway infrastructure.
The schedule generator will assign tasks to all the agent within the railway network.

The railway infrastructure can be build using any of the provided generators in env/rail_generators.py
Here we use the sparse_rail_generator with the following parameters:



In [ ]:
n_agents = 3 # Number of trains that have an assigned task in the env
# Characteristics of the flatland grid:
x_dim = 50 # With of map
y_dim = 50 # Height of map
n_cities = 4 # Number of cities where agents can start or end
max_rails_between_cities = 2 # Max number of tracks allowed between cities. This is number of entry point to a city
max_rails_in_city = 3 # Max number of parallel tracks within a city, representing a realistic trainstation

seed = 42 # Random seed

tree_observation = TreeObsForRailEnv(max_depth=2)

The schedule generator can make very basic schedules with a start point, end point and a speed profile for each agent.
The speed profiles can be adjusted directly as well as shown later on. We start by introducing a statistical distribution of speed profiles

In [ ]:
# Different agent types (trains) with different speeds.
speed_ration_map = {1.: 0.25,  # Fast passenger train
                    1. / 2.: 0.25,  # Fast freight train
                    1. / 3.: 0.25,  # Slow commuter train
                    1. / 4.: 0.25}  # Slow freight train

# We can now initiate the schedule generator with the given speed profiles
schedule_generator = sparse_line_generator(speed_ration_map)

In [ ]:
# We can furthermore pass stochastic data to the RailEnv constructor which will allow for stochastic malfunctions
# during an episode.

stochastic_data = MalfunctionParameters(malfunction_rate=1/10000,  # Rate of malfunction occurence
                                        min_duration=15,  # Minimal duration of malfunction
                                        max_duration=50  # Max duration of malfunction
                                        )



Construct the enviornment with the given observation, generataors, predictors, and stochastic data
After creating the environment, we need to call `reset()` to initialize it.

In [ ]:
env = RailEnv(width=x_dim,
              height=y_dim,
              rail_generator=sparse_rail_generator(
                  max_num_cities=n_cities,
                  seed=seed,
                  grid_mode=False,
                  max_rails_between_cities=max_rails_between_cities,
                  max_rail_pairs_in_city=max_rails_in_city
              ),
              line_generator=schedule_generator,
              number_of_agents=n_agents,
              malfunction_generator=ParamMalfunctionGen(stochastic_data),
              obs_builder_object=tree_observation,
              remove_agents_at_target=True,
              record_steps=True
              )

obs = env.reset()

The `RenderTool` allows us to visualize the environment.

In [ ]:
env_renderer = RenderTool(env, gl="PGL", screen_width=500, screen_height=500, show_debug=False)

We first look at the map we have created

In [ ]:
image = env_renderer.render_env(show=False, show_observations=False, show_inactive_agents=True, return_image=True)
plt.subplots(figsize=(12,12))
plt.imshow(image)

As an initial example we create a random agent

In [ ]:
# Now we can take 500 random steps in our environment and then visualize how the train moved in this episode.
class RandomAgent:

    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size

    def act(self, state):
        """
        :param state: input is the observation of the agent
        :return: returns an action
        """
        return np.random.choice([RailEnvActions.MOVE_FORWARD, RailEnvActions.MOVE_RIGHT, RailEnvActions.MOVE_LEFT,
                                 RailEnvActions.STOP_MOVING])

    def step(self, memories):
        """
        Step function to improve agent by adjusting policy given the observations

        :param memories: SARS Tuple to be
        :return:
        """
        return

    def save(self, filename):
        # Store the current policy
        return

    def load(self, filename):
        # Load a policy
        return


Initialize the agent with the parameters corresponding to the environment and observation_builder

In [ ]:
controller = RandomAgent(218, env.action_space[0])

We start by looking at the information of each agent.
We can see the task assigned to the agent by looking at:

In [ ]:
print("\n Agents in the environment have to solve the following tasks: \n")
for agent_idx, agent in enumerate(env.agents):
    print(
        "The agent with index {} has the task to go from its initial position {}, facing in the direction {} to its target at {}.".format(
            agent_idx, agent.initial_position, agent.direction, agent.target))

The agent will always have a status indicating if it is currently present in the environment or done or active.
For example we see that agent with index 0 is currently not active

In [ ]:
print("\n Their current statuses are:")
print("============================")

for agent_idx, agent in enumerate(env.agents):
    print("Agent {} status is: {} with its current position being {}".format(agent_idx, str(agent.state),
                                                                             str(agent.position)))

The agent needs to take any action [1,2,3] except do_nothing or stop to enter the level.

If the starting cell is free they will enter the level.
If multiple agents want to enter the same cell at the same time the lower index agent will enter first.

Let's check if there are any agents with the same start location:

In [ ]:
agents_with_same_start = set()
print("\n The following agents have the same initial position:")
print("=====================================================")
for agent_idx, agent in enumerate(env.agents):
    for agent_2_idx, agent2 in enumerate(env.agents):
        if agent_idx != agent_2_idx and agent.initial_position == agent2.initial_position:
            print("Agent {} as the same initial position as agent {}".format(agent_idx, agent_2_idx))
            agents_with_same_start.add(agent_idx)


Let's try to enter with all of these agents at the same time

In [ ]:
action_dict = dict()

for agent_id in agents_with_same_start:
    action_dict[agent_id] = 1  # Try to move with the agents

# Do a step in the environment to see what agents entered:
env.step(action_dict)

You will also notice, that the agents move at different speeds once they are on the rail.
The agents will always move at full speed when moving, never a speed inbetween.
The fastest an agent can go is 1, meaning that it moves to the next cell at every time step.

All slower speeds indicate the fraction of a cell that is moved at each time step.
Lets look at the current speed data of the agents:

In [ ]:
print("\n The speed information of the agents are:")
print("=========================================")

for agent_idx, agent in enumerate(env.agents):
    print(
        "Agent {} speed is: {:.2f} with the current fractional position being {}".format(
            agent_idx, agent.speed_counter.speed, agent.speed_counter.max_count))


New the agents can also have stochastic malfunctions happening which will lead to them being unable to move for a certain amount of time steps.
The malfunction data of the agents can easily be accessed as follows

In [ ]:
print("\n The malfunction data of the agents are:")
print("========================================")

for agent_idx, agent in enumerate(env.agents):
    print(
        "Agent {} is OK = {}".format(
            agent_idx, not agent.malfunction_handler.in_malfunction))


Now that you have seen these novel concepts that were introduced you will realize that agents don't need to take
an action at every time step as it will only change the outcome when actions are chosen at cell entry.

Therefore the environment provides information about what agents need to provide an action in the next step.

You can access this in the following way:

In [ ]:
# Chose an action for each agent
for a in range(env.get_num_agents()):
    action = controller.act(0)
    action_dict.update({a: action})
# Do the environment step
observations, rewards, dones, information = env.step(action_dict)
print("\n The following agents can register an action:")
print("========================================")
for info in information['action_required']:
    print("Agent {} needs to submit an action.".format(info))


Let us now look at an episode playing out with random actions performed

In [ ]:
print("\nStart episode...")

# Reset the rendering system
env_renderer.reset()

score = 0
# Run episode
frame_step = 0
frames = []

In [ ]:
for step in range(500):
    # Chose an action for each agent in the environment
    for a in range(env.get_num_agents()):
        action = controller.act(observations[a])
        action_dict.update({a: action})

    # Environment step which returns the observations for all agents, their corresponding
    # reward and whether their are done

    next_obs, all_rewards, done, _ = env.step(action_dict)

    frame = env_renderer.render_env(show=False, show_observations=False, show_inactive_agents=True, show_predictions=False, return_image=True)
    frames.append(frame)
    frame_step += 1
    # Update replay buffer and train agent
    for a in range(env.get_num_agents()):
        controller.step((observations[a], action_dict[a], all_rewards[a], next_obs[a], done[a]))
        score += all_rewards[a]

    observations = next_obs.copy()
    if done['__all__']:
        break
    print('Episode: Steps {}\t Score = {}'.format(step, score))


Our helper function `display_episode` allows us to get an animated episode. By putting `%%capture` at the begining of the cell we can surpress the cell output and create a clean output in the next cell.

In [ ]:
%%capture
animation = display_episode(frames)

In [ ]:
animation

## Baseline Challenge: Can a Rule-Based Agent Beat Random?

Before we use DQN, let's try a **rule-based agent** — no learning, just a fixed heuristic.

The simplest useful heuristic for Flatland: **always try to move forward**. Unlike the random agent that also stops and moves backwards, this agent keeps making progress toward the target.

**Your task:** Run both agents and compare their scores. Tomorrow with DQN, we'll beat both baselines.

In [ ]:
class RuleBasedAgent:
    """Always move forward — a simple but effective heuristic for train routing."""

    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size

    def act(self, state):
        return RailEnvActions.MOVE_FORWARD

    def step(self, memories):
        return

    def save(self, filename):
        return

    def load(self, filename):
        return

In [ ]:
def run_episode(agent, env, max_steps=500):
    """Run one episode with the given agent and return total score."""
    obs, info = env.reset()
    action_dict = {}
    score = 0

    for step in range(max_steps):
        for a in range(env.get_num_agents()):
            action_dict[a] = agent.act(obs[a])
        obs, rewards, done, _ = env.step(action_dict)
        score += sum(rewards.values())
        if done['__all__']:
            break

    return score

In [ ]:
n_eval = 3

random_agent = RandomAgent(218, env.action_space[0])
rule_agent = RuleBasedAgent(218, env.action_space[0])

print("Evaluating agents (this takes ~30 seconds)...")
random_scores = [run_episode(random_agent, env) for _ in range(n_eval)]
rule_scores = [run_episode(rule_agent, env) for _ in range(n_eval)]

print(f"\nRandom agent     | Mean score: {np.mean(random_scores):8.1f} +/- {np.std(random_scores):.1f}")
print(f"Rule-based agent | Mean score: {np.mean(rule_scores):8.1f} +/- {np.std(rule_scores):.1f}")

fig, ax = plt.subplots(figsize=(6, 4))
labels = ['Random', 'Rule-based\n(always forward)']
means = [np.mean(random_scores), np.mean(rule_scores)]
stds = [np.std(random_scores), np.std(rule_scores)]
ax.bar(labels, means, yerr=stds, capsize=6, color=['#e74c3c', '#3498db'], width=0.5)
ax.set_ylabel('Mean Episode Score')
ax.set_title('Flatland: Baseline Comparison')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

## What's Next? DQN (Notebook 04)

| Agent | Strategy | Learns from experience? | Adapts to new maps? |
|---|---|---|---|
| Random | Pick any action at random | No | No |
| Rule-based (always forward) | Fixed heuristic | No | No |
| **DQN** (Notebook 04) | Neural network trained on replay experience | **Yes** | **Yes** |

The rule-based agent beats random in open stretches — but it **cannot**:
- Detect that a train is blocking the track ahead
- Learn to pause and let a faster train pass to avoid deadlocks
- Coordinate across multiple agents

**DQN learns all of this from experience.** In Notebook 04 you'll load a pre-trained DQN agent on this same Flatland environment and see how far it outperforms both baselines here.